# Part 9 - The Durable Agent: event journal, replay, and effectively-once

*Agents from First Principles, Part 9.*

Everything so far lived in one fragile process. Pull the plug mid-run and the whole state, every observation, every decision, the half-finished refund, is gone. Worse, the obvious recovery is the dangerous one: just run the task again. If the refund already posted before the crash, a naive rerun posts it a SECOND time. The agent needs to survive a crash AND survive its own recovery.

Part 2 gave the refund tool a LOCAL idempotency guard: an in-memory dict that stops a retry inside one run from double-acting. That dict dies with the process. This part makes durability real with three pieces.

1. **An append-only event journal.** Every step writes an event (`run_started`, `llm_decided`, `tool_result`, `finished`) to a log keyed by a fixed `run_id`. The log is the source of truth, and STATE IS THE FOLD OVER THE LOG: to know what has happened, you replay the events, you do not trust in-memory variables. We keep it in a list here and print it as JSONL; a real system appends to a JSONL file or SQLite so it survives the process.

2. **Deterministic replay with step memoization.** On resume, fold the journal to see which steps already finished, and return their recorded results WITHOUT re-running them. Only the tail after the crash actually executes again.

3. **Idempotency keys for side-effecting tools.** Memoization alone is not enough, because of the worst-case crash: the refund POSTS and the provider records it, but the process dies BEFORE the `tool_result` is journaled. On replay the journal has no result for that step, so memoization would re-run it. The idempotency KEY saves us: the refund carries a stable key, the provider remembers keys it has already honored (a durable keystore), and a repeat of the same key returns the original result instead of charging again. That is EFFECTIVELY-ONCE across a replay, and it hardens Part 2's local guard.

This is an orthogonal DURABILITY layer the always-in-memory loop never had. It does not re-teach the reason/act/observe loop or the tool contract; it references Part 2's local guard (which this hardens) and Part 8's limits theme. The journal built here is REUSED by Part 10 (pause/approve/resume) and Part 11 (spans).

> **Runs with no network, no API key, and no third-party dependency.** The deterministic plan is the offline default; `generate()` is the real-LLM path one flag away. Set `OPENAI_API_KEY` to see the real banner; the code always falls through to the deterministic plan so output stays reproducible.

Companion script: `durable_agent.py`

## Setup, and what the durability claims actually mean

Three standard-library imports do the work: `os` lets us check for an API key without ever requiring one, `json` renders each event as a JSONL line, and `copy` lets us deep-copy the post-crash world so the naive restart and the durable resume each start from the same surviving state. Everything else is plain Python, so every cell runs fully offline, exactly as in Parts 1-8.

Be precise about the scope of the claims, because honesty matters more than a dramatic demo:

- **Byte-reproducible replay holds ONLY on the deterministic path.** With a real LLM you cache each decision in the journal and replay the cached decision rather than re-generating it (best-effort, not byte identical).
- **"Cross-process resume" here means rerun the same script on the same journal.** It is not true inter-process messaging; the journal is the artifact that would survive on disk.
- **The keystore models the payment provider's own idempotency.** That provider-side memory of honored keys is what makes the effectively-once guarantee real in practice, not a trick local to our code.

In [ ]:
import copy
import json
import os

print("ready -- no network, no API key, no third-party dependency required")

## A fixed run id and frozen timestamps

The journal is only reproducible if its identifiers are stable. `RUN_ID` is a fixed run id so every event ties back to the same run across a crash and a resume, and `TS_BASE` plus the sequence number gives each event a frozen timestamp (`TS_BASE + seq`). A real system would use a generated run id and real wall-clock timestamps; freezing them here is what lets two runs of this notebook produce byte-identical journals.

In [ ]:
RUN_ID = "run-7f3a"                       # a fixed run id so the journal is reproducible
TS_BASE = "2026-07-05T10:00:"             # frozen timestamps: TS_BASE + seq

print(f"run_id {RUN_ID!r}; timestamps frozen as {TS_BASE}NNZ")

## Step 1 - The event journal: append-only events, state is the FOLD over them

A "world" bundles the three durable artifacts that must survive a crash:

- `journal` -- the event log, append-only.
- `keystore` -- idempotency keys the provider has already honored.
- `ledger` -- the real side effect (money that actually moved).

`append(world, etype, data)` writes one event with the next sequence number, the fixed run id, a frozen timestamp, a type, and a payload. `replay(journal)` is the heart of the design: it folds the log into state, returning which step indices completed, their recorded results, and whether the run already finished. On resume this is the ONLY source of truth; we never trust in-memory variables that died with the crash. `print_journal` renders the log as sorted JSONL, which is what would survive on disk.

In [ ]:
def new_world():
    return {"journal": [], "keystore": {}, "ledger": {}}


def append(world, etype, data):
    seq = len(world["journal"])
    event = {"seq": seq, "run_id": RUN_ID, "ts": f"{TS_BASE}{seq:02d}Z",
             "type": etype, "data": data}
    world["journal"].append(event)
    return event


def replay(journal):
    """Fold the log into state: which step indices completed, their recorded
    results, and whether the run already finished. This is the ONLY source of
    truth on resume; we never trust in-memory variables that died with the crash."""
    completed, results, finished = set(), {}, False
    for e in journal:
        if e["type"] == "tool_result":
            completed.add(e["data"]["idx"])
            results[e["data"]["idx"]] = e["data"]["result"]
        elif e["type"] == "finished":
            finished = True
    return completed, results, finished


def print_journal(world):
    for e in world["journal"]:
        print("    " + json.dumps(e, sort_keys=True))


print("new_world, append, replay, print_journal ready.")

## Step 2 - The tools: a read-only search and an idempotent-by-key refund

`exec_search` is read-only: it returns the policy clause and a `False` flag meaning "no idempotency shortcut was taken." `exec_refund` is the side-effecting tool, and it is IDEMPOTENT BY KEY. The provider (the keystore) remembers every key it has honored, so the same key never charges twice no matter how many times it is replayed.

The crash-safety hinges on one ordering detail: the effect (money moving in the ledger) and the keystore write are modeled as one atomic step, so the key survives even a crash right here. The first time a key is seen, the money moves and the key is recorded; every later time the same key arrives, `exec_refund` returns the ORIGINAL result and reports `was_idem=True` instead of charging again. That is exactly the provider-side memory Part 2's local in-memory guard could not give us.

In [ ]:
def exec_search(args):
    return "Refunds after the window are refundable minus a 10% restocking fee.", False


def exec_refund(world, args, idem_key):
    ks = world["keystore"]
    if idem_key in ks:                                # the provider has seen this key
        return ks[idem_key], True                     # return the original; do NOT re-charge
    # The effect: money actually moves. The provider records the key atomically with
    # the charge (modeled here as one step), so the key survives even a crash here.
    world["ledger"][args["order_id"]] = world["ledger"].get(args["order_id"], 0.0) + args["amount"]
    result = f"refunded ${args['amount']:.2f} to {args['order_id']}"
    ks[idem_key] = result
    return result, False


print("exec_search, exec_refund ready.")

## The plan: two steps, with a stable idempotency key on the refund

`STEPS` is the fixed plan as `(tool, args, idempotency_key)` tuples: first a read-only `search_policy`, then the side-effecting `process_refund` for `ORD-3300` at `$180.00`. The refund's key, `ORD-3300:refund:180.00`, is STABLE across runs: it is derived from the order, the action, and the amount, not from anything that changes on a rerun. That stability is the whole point, because it is what lets the provider recognize a replayed refund as the same refund. `FINAL_ANSWER` is the message a completed run reports.

In [ ]:
STEPS = [
    ("search_policy", {"query": "refund policy window"}, None),
    ("process_refund", {"order_id": "ORD-3300", "amount": 180.0}, "ORD-3300:refund:180.00"),
]
FINAL_ANSWER = "Refund of $180.00 for ORD-3300 is complete (processed exactly once)."

print(f"{len(STEPS)} steps; refund idempotency key {STEPS[1][2]!r}")

## Why the crash is placed AFTER the effect but BEFORE the journaled result

This is the crux of the whole part, so it is worth stating plainly. The durable run loop journals an `llm_decided` event, then executes the tool (which moves money and records the key), then journals the `tool_result`. The crash is deliberately injected in the gap BETWEEN the effect and the `tool_result`:

1. `llm_decided` for the refund is journaled.
2. The refund effect runs: the ledger is touched and the key is honored in the keystore.
3. **The process is killed here** -- before the `tool_result` for the refund is ever written.

That ordering is what makes memoization alone fail. `replay()` learns which steps completed from `tool_result` events; with no `tool_result` for the refund, replay does NOT see it as completed, so memoization would happily re-run it. If memoization were the only defense, the resume would post the refund a second time, exactly the bug the naive restart demonstrates.

The idempotency key is the thing that saves the hard case. On resume the refund step re-executes, but the provider already has its stable key in the keystore, so `exec_refund` returns the original result with `was_idem=True` and never charges again. Memoization handles the EASY crash (one that lands after the `tool_result` is journaled); the idempotency key handles the HARD one (effect done, result not yet journaled).

**Honesty on scope.** The byte-reproducible journal you will see below holds only on this deterministic path. With a real LLM the decisions are cached into the journal and the cached decision is replayed (best-effort, not byte identical). "Cross-process resume" means rerunning the same script on the same surviving journal, and the keystore stands in for the payment provider's own idempotency, which is what makes the guarantee real in practice.

## Step 3 - The durable run loop, the naive restart, and a ledger formatter

`run_durable` is the loop that does all three durable things at once. It journals `run_started` on a fresh world, folds the journal with `replay()`, and short-circuits if the run already finished. For each step it skips completed steps (MEMOIZATION, printing the recorded result), otherwise journals `llm_decided`, executes the tool, and journals `tool_result`. The `crash_after` parameter injects the kill at exactly the gap described above: after the effect, before the `tool_result`.

`run_naive` is the dangerous recovery: it ignores the journal and idempotency entirely and just does the plan again, so the refund posts a SECOND time. `ledger_str` renders the ledger for the trace.

In [ ]:
def run_durable(world, crash_after=None, label="run"):
    if not world["journal"]:
        append(world, "run_started", {"run_id": RUN_ID})
    completed, results, finished = replay(world["journal"])
    if finished:
        print(f"    [{label}] journal shows the run already finished; nothing to do.")
        return results
    for idx, (tool, args, idem) in enumerate(STEPS):
        if idx in completed:                          # MEMOIZED: do not re-run
            print(f"    step {idx} {tool}: memoized from journal -> {results[idx]!r}")
            continue
        append(world, "llm_decided", {"idx": idx, "tool": tool, "args": args})
        if tool == "process_refund":
            result, was_idem = exec_refund(world, args, idem)
        else:
            result, was_idem = exec_search(args)
        if crash_after == idx:                        # *** the kill ***
            print(f"    step {idx} {tool}: effect done (ledger touched, key honored), but")
            print(f"    *** PROCESS KILLED before the result was journaled ***")
            return results
        tag = " (idempotent: key already honored, no second charge)" if was_idem else ""
        append(world, "tool_result", {"idx": idx, "tool": tool, "result": result})
        print(f"    step {idx} {tool}: {result}{tag}")
    append(world, "finished", {"answer": FINAL_ANSWER})
    print(f"    finished -> {FINAL_ANSWER}")
    return results


def run_naive(world):
    """A restart that ignores the journal and idempotency entirely: just do the plan
    again. This is the dangerous recovery: the refund posts a SECOND time."""
    for idx, (tool, args, idem) in enumerate(STEPS):
        if tool == "process_refund":
            world["ledger"][args["order_id"]] = world["ledger"].get(args["order_id"], 0.0) + args["amount"]
            print(f"    step {idx} {tool}: refunded ${args['amount']:.2f} to {args['order_id']} "
                  "(no idempotency check!)")
        else:
            print(f"    step {idx} {tool}: re-ran from scratch")


def ledger_str(world):
    return ", ".join(f"{k}=${v:.2f}" for k, v in world["ledger"].items()) or "(empty)"


print("run_durable, run_naive, ledger_str ready.")

## Step 4 - `generate()`: the real LLM path (reference shape only)

Same device as Parts 1-8. On the real path each decision is CACHED into the journal and replayed from there, not re-generated, which is why the byte-reproducibility claim is scoped to the deterministic plan. The offline demo never calls `generate()`; the deterministic `STEPS` plan is the source of truth for everything in the output. SDK names and model IDs move fast, so only `generate()` would need edits to go live.

In [ ]:
def generate(prompt):
    """REAL path: ask a hosted LLM for the next step; cache the decision in the
    journal so replay is faithful. Unused offline."""
    from openai import OpenAI
    client = OpenAI()                               # reads OPENAI_API_KEY
    resp = client.chat.completions.create(
        model="gpt-4o-mini",                        # a small chat model; check names
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

# Anthropic / Claude alternative. Swap in for generate() above:
#
# def generate(prompt):
#     from anthropic import Anthropic
#     client = Anthropic()                            # reads ANTHROPIC_API_KEY
#     resp = client.messages.create(
#         model="claude-opus-4-8",                    # check current model names
#         max_tokens=1024,
#         messages=[{"role": "user", "content": prompt}],
#     )
#     return resp.content[0].text


if os.environ.get("OPENAI_API_KEY"):
    print("[controller] OPENAI_API_KEY set; the real LLM path caches each decision in the "
          "journal and replays it. Falling through to the deterministic plan for reproducibility.")
else:
    print("[controller] no OPENAI_API_KEY; using the deterministic plan (offline default)")

## Demo - the crash, the naive double-charge, then the durable resume

Everything below runs fully offline. The three sections share one surviving post-crash world (deep-copied so the naive restart and the durable resume each start from the same state):

1. **Run 1 crashes mid-refund.** The refund posts (the ledger shows `$180.00`) but the process dies before the `tool_result` is journaled, so the journal has no result for step 1.
2. **The naive restart.** It ignores the journal and idempotency and re-runs the plan, posting the refund a second time: the ledger jumps to `$360.00`. This is the bug.
3. **The durable resume.** It replays the journal (step 0 is memoized), re-attempts the refund whose result was never journaled, and the idempotency key prevents a second charge. The ledger stays at `$180.00`. Effectively-once.

In [ ]:
bar = "=" * 72
print(bar)
print("THE DURABLE AGENT  -  event journal, replay, and effectively-once")
print(bar)
if os.environ.get("OPENAI_API_KEY"):
    print("[controller] OPENAI_API_KEY set; the real LLM path caches each decision in the "
          "journal and replays it. Falling through to the deterministic plan for reproducibility.")
else:
    print("[controller] no OPENAI_API_KEY; using the deterministic plan (offline default)")

### 1) Run 1 crashes after the refund posts but before it is journaled

The durable loop runs with `crash_after=1`: step 0 (search) journals cleanly, then step 1 (the refund) executes its effect and is killed in the gap before its `tool_result` is written. The journal that survives has four events, ending at the refund's `llm_decided` with NO `tool_result` for step 1. The ledger already shows `$180.00`: the refund DID post. This is precisely the state where memoization alone would re-run the refund, because replay cannot see step 1 as completed.

In [ ]:
print("\n" + "-" * 72)
print("1) RUN 1 crashes after the refund posts but before it is journaled.")
print("-" * 72)
world = new_world()
run_durable(world, crash_after=1, label="run-1")
print(f"\n  Journal after the crash (this is what survives on disk):")
print_journal(world)
print(f"  Ledger after the crash: {ledger_str(world)}  (the refund DID post)")
print("  Note: the journal has NO tool_result for step 1, so memoization alone would re-run it.")

### 2) The naive restart: a double charge

`run_naive` is the recovery a system without durability would do: just run the plan again from scratch, ignoring both the journal and the keystore. It re-runs step 0 and then posts the refund a SECOND time on top of the one that already survived the crash. The ledger jumps from `$180.00` to `$360.00`. This is the customer-facing bug the rest of the part exists to prevent.

In [ ]:
print("\n" + bar)
print("2) NAIVE RESTART (ignores the journal and idempotency): double charge.")
print(bar)
naive = copy.deepcopy(world)                       # same post-crash world (ledger has 1 refund)
run_naive(naive)
print(f"    ledger now: {ledger_str(naive)}  <- DOUBLE REFUND ($360.00). This is the bug.")

### 3) The durable resume: replay + memoization + the idempotency key

Now resume from the SAME surviving post-crash world, but durably: `run_durable` with `crash_after=None`. Replay memoizes step 0 straight from the journal (no re-run). Step 1's `tool_result` was never journaled, so the refund is re-attempted, but the provider already holds its stable key, so `exec_refund` returns the original result with `was_idem=True` and does NOT charge again. The run then finishes.

The journal grows to seven events (seq 0 through 6). Note the honest detail: there are TWO `llm_decided` events for step 1 (seq 3 from run 1, seq 4 from the resume), because the decision is re-made on resume. The EFFECT is not re-made: the ledger stays at `$180.00`. Effectively-once.

In [ ]:
print("\n" + bar)
print("3) DURABLE RESUME (replay the journal; the idempotency key prevents a re-charge).")
print(bar)
resumed = copy.deepcopy(world)                     # same post-crash world (ledger has 1 refund)
run_durable(resumed, crash_after=None, label="resume")
print(f"\n  Journal after the resume:")
print_journal(resumed)
print(f"  Ledger after the resume: {ledger_str(resumed)}  <- still ONE refund. Effectively-once.")

print("\n" + bar)
print("Done. Durability is three things working together:")
print("  - an append-only JOURNAL where state is the FOLD over the log (survives the crash)")
print("  - deterministic REPLAY with step MEMOIZATION (only the tail re-runs)")
print("  - IDEMPOTENCY KEYS so a re-run of a side effect returns the original, never doubles")
print("Memoization handles the easy crash; the idempotency key handles the hard one")
print("(effect done, result not yet journaled). Part 2's local guard, now effectively-once.")
print(bar)

## Wrap-up

Part 8 made the agent safe to leave running: budgets, a loop detector, and a circuit breaker that stops a runaway with a graceful partial result. But every run still lived and died in one process. Crash midway and the transcript, the budget, and the half-finished refund were gone, and the obvious recovery, just run it again, re-charged the customer.

Part 9 added an orthogonal durability layer on three legs:

- **The append-only event journal**, where state is the FOLD over the log. To know what happened you replay the events; you never trust in-memory variables that died with the crash. We kept it in a list and printed it as JSONL, but a real system appends to a file or SQLite so it survives.
- **Deterministic replay with step memoization.** On resume, completed steps return their recorded results without re-running; only the tail after the crash executes again.
- **Idempotency keys for side-effecting tools.** The hard crash (effect done, `tool_result` not yet journaled) defeats memoization alone, but the stable key lets the provider's durable keystore return the original result and never charge twice. That is effectively-once, and it hardens Part 2's local guard.

Remember the scope: byte-reproducible replay holds only on the deterministic path; with a real LLM you cache each decision in the journal and replay the cached decision. "Cross-process resume" means rerunning the same script on the same surviving journal, and the keystore models the payment provider's own idempotency.

**Part 10 - pause, approve, resume** is next. The journal built here is exactly what makes a human-in-the-loop checkpoint possible: a durable run can pause before a risky action, wait for approval, and resume from the log without redoing the work or re-charging the customer. Part 11 reuses the same journal again, this time as the backbone for spans and tracing.